In [ ]:
import pandas as pd
import numpy as np

orders   = pd.read_csv("https://cdn.enqurious.com/documents/660a051c-25ef-4fd2-96bc-7179565c8d6f_exorders.csv")
returns  = pd.read_csv("https://cdn.enqurious.com/documents/e4961106-208d-4b54-bb13-f9dca0cddf7f_exreturns.csv")
products = pd.read_csv("https://cdn.enqurious.com/documents/bf35fdf1-d9a0-4432-9b55-6a8cb2720716_exproducts.csv")
transactions = pd.read_csv("https://cdn.enqurious.com/documents/d2628fca-f225-4482-80ee-889fdf3f95aa_extransactions.csv")

# Your code here

In [ ]:
import pandas as pd
import numpy as np

# ==============================================================================
# STEP 1: Load the Required Datasets
# ==============================================================================
# Read all the datasets needed for the analysis.
# Orders contain customer purchases.
# Transactions connect orders with products.
# Products provide category information.
# Returns identify which orders were returned.
orders = pd.read_csv("https://cdn.enqurious.com/documents/660a051c-25ef-4fd2-96bc-7179565c8d6f_exorders.csv")
returns = pd.read_csv("https://cdn.enqurious.com/documents/e4961106-208d-4b54-bb13-f9dca0cddf7f_exreturns.csv")
products = pd.read_csv("https://cdn.enqurious.com/documents/bf35fdf1-d9a0-4432-9b55-6a8cb2720716_exproducts.csv")
transactions = pd.read_csv("https://cdn.enqurious.com/documents/d2628fca-f225-4482-80ee-889fdf3f95aa_extransactions.csv")

# ==============================================================================
# STEP 2: Link Orders with Products
# ==============================================================================
# The Orders table does not contain Product IDs directly.
# Use the Transactions table to identify which products belong to each order.
# Remove duplicate Order-Product combinations if they exist.
order_product = (
    transactions[['Order_ID', 'Product_ID']]
    .drop_duplicates()
)

# ==============================================================================
# STEP 3: Attach Product Category Information
# ==============================================================================
# Merge the Product IDs with the Products table
# to retrieve the category for each ordered product.
order_category = (
    order_product
    .merge(
        products[['product_id', 'category']],
        left_on='Product_ID',
        right_on='product_id',
        how='left'
    )
)

# ==============================================================================
# STEP 4: Combine Category Information with Orders
# ==============================================================================
# Add the product category to each order so every order
# can be analyzed by its category.
orders_with_category = (
    orders
    .merge(
        order_category[['Order_ID', 'category']],
        left_on='order_id',
        right_on='Order_ID',
        how='left'
    )
)

# ==============================================================================
# STEP 5: Identify Returned Orders
# ==============================================================================
# Create a Boolean flag indicating whether each order
# appears in the Returns dataset.
orders_with_category['is_return'] = (
    orders_with_category['order_id']
    .isin(returns['order_id'])
)

# ==============================================================================
# STEP 6: Calculate Category-wise Return Statistics
# ==============================================================================
# For each product category:
# • Count the total number of orders.
# • Count how many of those orders were returned.
summary = (
    orders_with_category
    .groupby('category')
    .agg(
        total_orders=('order_id', 'count'),
        total_returns=('is_return', 'sum')
    )
    .reset_index()
)

# ==============================================================================
# STEP 7: Calculate Return Rate
# ==============================================================================
# Return Rate = (Returned Orders / Total Orders) × 100
# Format the result as a percentage with two decimal places.
summary['return_rate'] = (
    summary['total_returns'] /
    summary['total_orders'] * 100
).map('{:.2f}%'.format)

# Convert returned order count back to integer
# because summing Boolean values produces numeric values.
summary['total_returns'] = summary['total_returns'].astype(int)

# ==============================================================================
# STEP 8: Display the Final Summary
# ==============================================================================
# Display category-wise order count, return count,
# and return percentage.
print(summary.to_string(index=False))

       category  total_orders  total_returns return_rate
      Furniture          2082            172       8.26%
Office Supplies          5903            470       7.96%
     Technology          1782            148       8.31%


In [ ]:
# ==============================================================================
# STEP 9: Create a Pivot Table
# ==============================================================================
# Convert the summary DataFrame into a pivot table.
# Here:
# • Each product category becomes a row.
# • Total orders, total returns, and return rate become columns.
# • aggfunc='first' is used because there is only one record per category,
#   so no aggregation is actually required.
pivot_df = (
    summary
    .pivot_table(
        index='category',
        values=['total_orders', 'total_returns', 'return_rate'],
        aggfunc='first'
    )
    .reset_index()
)

# ==============================================================================
# STEP 10: Display the Pivot Table
# ==============================================================================
# Display the category-wise summary in pivot table format.
print(pivot_df)

          category return_rate  total_orders  total_returns
0        Furniture       8.26%          2082            172
1  Office Supplies       7.96%          5903            470
2       Technology       8.31%          1782            148


In [ ]:
# ==============================================================================
# STEP 11: Convert the Pivot Table into Long Format
# ==============================================================================
# Transform the data from wide format to long format.
#
# Before Melt (Wide Format):
# ----------------------------------------------------
# Category     Total Orders   Total Returns   Return Rate
# Electronics       120              10          8.33%
# Furniture          80               5          6.25%
#
# After Melt (Long Format):
# ----------------------------------------------------
# Category      Metric            Value
# Electronics   total_orders      120
# Electronics   total_returns      10
# Electronics   return_rate      8.33%
# Furniture     total_orders       80
# Furniture     total_returns       5
# Furniture     return_rate      6.25%
#
# 'category' remains unchanged.
# The selected columns become two new columns:
# • metric → Stores the original column names.
# • value  → Stores the corresponding values.
long_df = (
    pivot_df
    .melt(
        id_vars='category',
        value_vars=['total_orders', 'total_returns', 'return_rate'],
        var_name='metric',
        value_name='value'
    )
)

# ==============================================================================
# STEP 12: Sort the Final Output
# ==============================================================================
# Arrange the records by category and metric
# to make the output easier to read.
long_df = (
    long_df
    .sort_values(['category', 'metric'])
    .reset_index(drop=True)
)

# ==============================================================================
# STEP 13: Display the Long Format Data
# ==============================================================================
# Display the transformed dataset.
print(long_df)

          category         metric  value
0        Furniture    return_rate  8.26%
1        Furniture   total_orders   2082
2        Furniture  total_returns    172
3  Office Supplies    return_rate  7.96%
4  Office Supplies   total_orders   5903
5  Office Supplies  total_returns    470
6       Technology    return_rate  8.31%
7       Technology   total_orders   1782
8       Technology  total_returns    148


In [ ]:
# ==============================================================================
# STEP 6: Calculate Category-wise Order Summary
# ==============================================================================
# Group the data by product category and compute:
# • Total number of orders in each category.
# • Total number of returned orders.
#
# Since 'is_return' is a Boolean column (True/False),
# sum() counts the number of True values (returned orders).
summary = (
    orders_with_category
    .groupby('category')
    .agg(
        total_orders=('order_id', 'count'),
        total_returns=('is_return', 'sum')
    )
    .reset_index()
)

# ==============================================================================
# STEP 7: Calculate Return Rate
# ==============================================================================
# Return Rate = (Returned Orders / Total Orders) × 100
# Format the result as a percentage with two decimal places.
summary['return_rate'] = (
    summary['total_returns'] /
    summary['total_orders'] * 100
).map('{:.2f}%'.format)